In [ ]:
import pandas as pd
from dask.array import average
from gototile.grid import SkyGrid
from goto_sim.scheduling import HEATSurvey
from goto_sim.utils import load_tilelist
from astropy.time import Time, TimeDelta
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import glob
import matplotlib.font_manager

plt.rcParams['font.family'] = 'DejaVu Sans'

In [ ]:
data_dir = "sims"
figure_dir = "figures"
file_pattern = f"{data_dir}/simulation_results_ha_limit_*.csv"
file_list = glob.glob(file_pattern)
if not file_list:
    raise FileNotFoundError(f"No files found matching pattern: {file_pattern}")
print(f"Found {len(file_list)} files matching the pattern.")
ha_lims = [float(f.split('_')[-1].replace('h.csv', '')) for f in file_list]
print(ha_lims)
results = {}
for file_path, ha_limit in zip(file_list, ha_lims):
    df = pd.read_csv(file_path, comment='#')
    results[ha_limit] = df
results_df = pd.concat(results, names=['ha_limit', 'index']).reset_index(level='ha_limit')
results_df['ha_hours'] = results_df['hour_angle'] / 15.0
heats_survey = HEATSurvey()
heats_tiles = heats_survey.tiles
cold_survey = heats_survey.generate_colds()
cold_tiles = cold_survey.get_tiles()
grid: SkyGrid = SkyGrid.from_name("GOTO")
results_df

In [ ]:
# Replace HEATS target tiles with HEATS, COLDs tiles with COLDs.
# This is fairly slow, probably a faster way to do this with mapping
results_df['target'] = results_df['target'].replace({tile: 'HEATS' for tile in heats_tiles})
results_df['target'] = results_df['target'].replace({tile: 'COLDs' for tile in cold_tiles})
results_df

In [ ]:
heats_results = results_df[results_df['target'] == 'HEATS']
cold_results = results_df[results_df['target'] == 'COLDs']

In [ ]:
heats_results.hist(column='ha_hours', by='ha_limit', bins=20, figsize=(12, 8))
heats_results.hist(column='altitude', by='ha_limit', bins=20, figsize=(12, 8))

In [ ]:
cadence = results_df.groupby(['tile', 'ha_limit'])['time'].apply(
    lambda x: np.mean(np.diff(np.sort(x)))
).sort_index()
print(f"Total results: {len(cadence)}")
cold_cadences = cadence.loc[cold_tiles]
print(f"COLD results: {len(cold_cadences)}")

heats_cadences = cadence.loc[heats_tiles]
print(f"HEATS results: {len(heats_cadences)}")

observations_per_tile = results_df.groupby(['ha_limit', 'tile']).size()
observations_per_tile = observations_per_tile.reindex(grid.tilenames, level='tile', fill_value=0)
observations_per_tile


In [ ]:
# Plot the average altitude of observations for each tile
average_altitude = results_df.groupby(['ha_limit', 'tile'])['altitude'].mean()
for (ha_limit, group_altitude) in average_altitude.groupby('ha_limit'):
    fig = plt.figure(figsize=(10, 6), dpi=100)
    fig.suptitle(f'Hour Angle Limit: {ha_limit:.2f} Hours - Average Altitude of Observations')
    gs = GridSpec(1, 2, height_ratios=[1], width_ratios=[3, 1])
    axes = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
    axes.grid()
    tiles = grid.plot_tiles(axes, array=group_altitude.values, cmap='viridis',
                            ec='none', alpha=0.9, zorder=0.6)
    grid.plot_tiles(axes, tilenames=heats_tiles, fc='none', ec='red', lw=0.5, zorder=1.6)
    grid.plot_tiles(axes, tilenames=cold_tiles, fc='none', ec='blue', lw=0.3, zorder=1.5)
    altitude_ticks = np.arange(0, 91, 10)
    fig.colorbar(
        tiles, ax=axes, ticks=altitude_ticks,
        fraction=0.1, pad=0.05, aspect=50, orientation='horizontal',
        label=f'Average altitude of observations (degrees)'
        f'\n(min={min(group_altitude.values):.1f}, '
        f'mean={np.mean(group_altitude.values):.1f}, '
        f'max={max(group_altitude.values):.1f})'
    )
    axes = fig.add_subplot(gs[0, 1])
    heats_altitudes = group_altitude.loc[:,heats_tiles]
    cold_altitudes = group_altitude.loc[:,cold_tiles]
    counts, bins, patches = axes.hist([heats_altitudes, cold_altitudes],
            bins=np.arange(30, 91, 5), stacked=True,
            color=['red', 'blue'],
            label=['Tiles in the HEATS survey (n={})'.format(len(heats_tiles)),
                   'Tiles in the COLD survey (n={})'.format(len(cold_tiles))],
    )
    axes.legend()
    axes.set_xlabel('Average alt(degrees)')
    axes.set_ylabel('NObservations')
    axes.set_xlim(30, 90)

    plt.savefig(f"{figure_dir}/ha_limit_{ha_limit}h_average_altitude.png", dpi=300)
    plt.show()

In [ ]:
# for each reserve fraction, plot the cadence and total number of observations for each tile

for (ha_limit, group_cadence) in cadence.groupby('ha_limit'):
    group_observations = observations_per_tile.loc[ha_limit]
    heats_observations = group_observations.loc[heats_tiles]
    cold_observations = group_observations.loc[cold_tiles]
    print(f"Hour Angle Limit: {ha_limit:.2f}")
    if True:
        fig = plt.figure(figsize=(12, 8), dpi=100)
        title = f'Hour Angle Limit: ±{ha_limit:.2f} Hour'
        if ha_limit != 1:
            title += 's'
        fig.suptitle(title)
        gs = GridSpec(2, 2, width_ratios=[2, 1], height_ratios=[1, 1])

        # -------------------------
        axes = fig.add_subplot(gs[0, 0], projection='astro hours mollweide')
        axes.grid()

        tiles = grid.plot_tiles(axes, array=group_observations.values, cmap='viridis',
                                ec='none', alpha=0.9, zorder=0.6)
        grid.plot_tiles(axes, tilenames=heats_tiles, fc='none', ec='red', lw=0.5, zorder=1.6)
        grid.plot_tiles(axes, tilenames=cold_tiles, fc='none', ec='blue', lw=0.3, zorder=1.5)
        count_ticks = np.arange(round(min(group_observations.values)-10, -1) if min(group_observations)> 20 else 0, max(group_observations.values) + 1, round(max(group_observations.values)//5, -1) if max(group_observations.values) > 100 else 2)
        fig.colorbar(
            tiles, ax=axes, ticks=count_ticks,
            fraction=0.1, pad=0.05, aspect=50, orientation='horizontal',
            label=f'Total number of observations'
            f'\n(min={min(group_observations.values):.0f}, '
            f'mean={np.mean(group_observations.values):.1f}, '
            f'max={max(group_observations.values):.0f})'
        )
        # ------------------------
        axes = fig.add_subplot(gs[1, 0], projection='astro hours mollweide')
        axes.grid()
        tiles = grid.plot_tiles(axes, array=group_cadence.values, cmap='viridis',
                                ec='none', alpha=0.9, zorder=0.6)
        grid.plot_tiles(axes, tilenames=np.array(grid.tilenames)[np.array(group_cadence.values) == 0],
                        fc='0.4', ec='none', zorder=0.7)

        grid.plot_tiles(axes, tilenames=heats_tiles, fc='none', ec='red', lw=0.5, zorder=1.6)
        grid.plot_tiles(axes, tilenames=cold_tiles, fc='none', ec='blue', lw=0.3, zorder=1.5)

        cadence_ticks = np.arange(0, int(max(group_cadence.values))+2, (max(group_cadence.values)//5 if max(group_cadence.values) > 5 else 1))
        fig.colorbar(
            tiles, ax=axes, ticks=cadence_ticks,
            fraction=0.1, pad=0.05, aspect=50, orientation='horizontal',
            label='Median days between observations'
            f'\n(min={min(group_cadence.values[group_cadence.values > 0]):.1f}, '
            f'mean={np.mean(group_cadence.values[group_cadence.values > 0]):.1f}, '
            f'max={max(group_cadence.values):.1f})'
        )
        # -------------------------
        axes = fig.add_subplot(gs[0, 1])
        counts, bins, patches = axes.hist([heats_observations, cold_observations],
                bins=np.arange(0, int(max(group_observations.values))+1, 10), stacked=True,
                color=['red', 'blue'],
                label=['Tiles in the HEATS survey (n={})'.format(len(heats_tiles)),
                       'Tiles in the COLD survey (n={})'.format(len(cold_tiles))],
        )
        axes.legend(loc='upper right', bbox_to_anchor=(1, 1.25), frameon=False)
        axes.set_xlabel('Number of observations')
        axes.set_ylabel('Frequency')
        axes.set_xlim(int(min(group_observations.values))-1, int(max(group_observations.values))+1+0.5)
        axes.set_xticks(count_ticks)
        # -------------------------
        axes = fig.add_subplot(gs[1, 1])
        #axes.hist([median_cadence[heats_mask]*24, median_cadence[~heats_mask]*24],
        axes.hist([group_cadence.loc[heats_tiles], group_cadence.loc[cold_tiles]],
                bins=np.arange(0, max(group_cadence.values)+1, 0.25), stacked=True,
                color=['red', 'blue'],
                #label=['Tiles in the HEATS survey', 'Tiles in the COLD survey'],
        )
        #axes.legend()
        axes.set_xlabel('Days between observations')
        axes.set_ylabel('Frequency')
        axes.set_xlim(0-0.1, int(max(group_cadence.values))+2)
        axes.set_xticks(cadence_ticks)
        plt.savefig(f"{figure_dir}/ha_limit_{ha_limit}h_summary.png", dpi=300)
        plt.show()


In [ ]:
# Split results up by month and plot average altitude histograms
results_df['astropytime'] = results_df['time'].apply(lambda x: Time(x, format='mjd'))
results_df['month'] = results_df['astropytime'].apply(lambda x: x.datetime.month)
months = results_df['month'].unique()
# Plot a violin plot of altitude distributions for each month and hour angle limit
for ha_limit, group in results_df.groupby('ha_limit'):
    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
    data = [group[group['month'] == month]['altitude'] for month in range(1, 13)]
    ax.violinplot(data, positions=range(1, 13), showmeans=True)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
    ax.set_xlabel('Month')
    ax.set_ylabel('Altitude (degrees)')
    ax.set_title(f'Hour Angle Limit: {ha_limit:.2f} Hours - Altitude Distribution by Month')
    plt.savefig(f"{figure_dir}/ha_limit_{ha_limit}h_monthly_altitude_violin.png", dpi=300)
    plt.show()